# XS-PH B1-V3 — تنفيذ تجربة عزل أثر Page–Hinkley

هذا Notebook ينفذ تجربة B1-V3 على Kaggle عبر مراحل منفصلة:

1. تحميل حزمة الأكواد إلى مساحة قابلة للكتابة في `/kaggle/working`.
2. التحقق من سلامة الملفات والجداول العشوائية.
3. تشغيل تطوير معاملات PH واختيار أفضل K ثابت من بيانات التطوير فقط.
4. حساب حجم عينة الاختبار من الفروق الزوجية في التطوير.
5. تشغيل اختبار نهائي ببذور مستقلة وإعدادات مجمدة.
6. تحليل الأداء، الحدود الزمنية، PFA الفعلي، AUC، وفواصل الثقة.

**تنبيه علمي:** لا تضمن هذه التجربة تفوق XS-PH. لا تعدّل معاملات PH أو K بعد الاطلاع على نتائج الاختبار النهائي. إذا لم تدعم النتائج التحسن، يجب الإبلاغ عن ذلك كما هو.

## تجهيز Kaggle

حمّل `XS_PH_B1_V3_KAGGLE_PACKAGE.zip` وأضفه إلى Kaggle كـ Dataset جديد، ثم اربطه بهذا Notebook من **Add Data**. سيبحث Notebook عن الحزمة تلقائياً داخل `/kaggle/input`. كما يمكن ربط مجلد الحزمة بعد فك ضغطه.

التجربة لا تحتاج GPU. تأكد من أن مساحة `/kaggle/working` كافية لحفظ ملفات النتائج، ولا تكتب داخل `/kaggle/input` لأنه للقراءة فقط.

In [ ]:
from pathlib import Path
import os, sys, shutil, zipfile, subprocess, json

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
PKG = WORK_ROOT / "XS_PH_B1_V3_PACKAGE"

REQUIRED = [
    "run_xs_study_v2.py", "run_b1v3_development.py",
    "select_b1v3_development.py", "run_b1v3_final.py",
    "analyze_b1v3.py", "validate_b1v3.py", "xs_experiment_v2.py",
    "xs_filter_v2.py", "requirements.txt", "README_B1_V3_AR.md"
]

def is_package(folder):
    return folder.is_dir() and all((folder / name).is_file() for name in REQUIRED)

candidates = []
if INPUT_ROOT.exists():
    for p in INPUT_ROOT.rglob("run_b1v3_development.py"):
        if is_package(p.parent):
            candidates.append(p.parent)

if not candidates and INPUT_ROOT.exists():
    zip_candidates = list(INPUT_ROOT.rglob("XS_PH_B1_V3_KAGGLE_PACKAGE.zip"))
    if zip_candidates:
        extract_dir = WORK_ROOT / "_b1v3_package_extract"
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True)
        with zipfile.ZipFile(zip_candidates[0]) as zf:
            zf.extractall(extract_dir)
        for p in extract_dir.rglob("run_b1v3_development.py"):
            if is_package(p.parent):
                candidates.append(p.parent)

if not candidates and is_package(PKG):
    PKG_SRC = PKG
else:
    if not candidates:
        raise FileNotFoundError(
            "B1-V3 package was not found. Add XS_PH_B1_V3_KAGGLE_PACKAGE.zip "
            "as a Kaggle Dataset and attach it with Add Data."
        )
    # Prefer the first complete B1-V3 package discovered under Kaggle input.
    PKG_SRC = candidates[0]

if PKG_SRC.resolve() != PKG.resolve():
    if PKG.exists():
        shutil.rmtree(PKG)
    shutil.copytree(PKG_SRC, PKG)

assert is_package(PKG), f"Incomplete writable package: {PKG}"
sys.path.insert(0, str(PKG))
print("Writable package:", PKG)
print("Package source:", PKG_SRC)
print("All required files found.")

## 1. تثبيت المتطلبات والتحقق البرمجي

يتم نسخ الحزمة أولاً إلى `/kaggle/working`؛ لذلك لن يحاول Python إنشاء `__pycache__` داخل مجلد Dataset المقروء فقط.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PKG / "requirements.txt")],
    check=True
)

validation = subprocess.run(
    [sys.executable, str(PKG / "validate_b1v3.py")],
    cwd=str(PKG),
    env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"},
    capture_output=True,
    text=True,
)
print(validation.stdout)
if validation.stderr:
    print(validation.stderr)
if validation.returncode != 0:
    raise RuntimeError("B1-V3 validation failed; do not run simulations.")

validation_report = json.loads((PKG / "b1v3_validation_report.json").read_text())
assert validation_report["status"] == "PASS"
print("B1-V3 validation PASS")

## 2. اختيار مرحلة التشغيل

غيّر `STAGE` في الخلية التالية إلى واحدة من: `validate` أو `pilot` أو `development` أو `final`.

- `validate`: يتحقق من الحزمة فقط.
- `pilot`: اختبار هندسي صغير لمسار التطوير، ولا يستخدم كدليل بحثي.
- `development`: ينفذ خمسة مرشحين لـ PH ويحسب Fixed-K مرة واحدة، ثم يختار PH وK من بيانات التطوير ويحسب خطة حجم العينة.
- `final`: يستخدم ملف الاختيار المجمد الموجود من مرحلة development، ويشغل مجموعات معايرة واختبار مستقلة ثم يحلل النتائج.

**لا تشغّل `final` قبل مراجعة ملف الاختيار وخطة حجم العينة.**

In [ ]:
# اختر مرحلة واحدة في كل مرة: validate / pilot / development / final
# 1) ابدأ بـ validate.
# 2) شغّل development لإجراء تطوير PH واختيار K وحساب حجم العينة.
# 3) بعد مراجعة B1V3_selection.json وخطة القوة، غيّر إلى final لتشغيل الاختبار المستقل وتحليله.
STAGE = "validate"

if STAGE not in {"validate", "pilot", "development", "final"}:
    raise ValueError("STAGE must be validate, pilot, development, or final")

RUN_DEV_PILOT = (STAGE == "pilot")
RUN_FULL_DEVELOPMENT = (STAGE == "development")
RUN_SELECTION = (STAGE == "development")
RUN_FINAL_TEST = (STAGE == "final")
RUN_FINAL_ANALYSIS = (STAGE == "final")

PILOT_OUT = WORK_ROOT / "B1V3_development_pilot"
DEV_OUT = WORK_ROOT / "B1V3_development_full"
FINAL_OUT = WORK_ROOT / "B1V3_final"

FINAL_SCORE_CAL_COUNT = 150
FINAL_TAU_CAL_COUNT = 150

print({
    "STAGE": STAGE,
    "RUN_DEV_PILOT": RUN_DEV_PILOT,
    "RUN_FULL_DEVELOPMENT": RUN_FULL_DEVELOPMENT,
    "RUN_SELECTION": RUN_SELECTION,
    "RUN_FINAL_TEST": RUN_FINAL_TEST,
    "RUN_FINAL_ANALYSIS": RUN_FINAL_ANALYSIS,
    "PILOT_OUT": str(PILOT_OUT),
    "DEV_OUT": str(DEV_OUT),
    "FINAL_OUT": str(FINAL_OUT),
})

## 3. (اختياري) Pilot development

يشغّل المرشحين الستة بإعدادات مصغرة: 4 بذور تطوير و12 بذرة لكل مجموعة معايرة، مع الإبقاء على 200 جولة كي تبقى الجلسات المتغيرة ذات معنى. هذه المرحلة لفحص مسار التطوير فقط. لا تستخدم `B1V3_selection.json` الناتج عنها لإطلاق الاختبار النهائي.

In [ ]:
if RUN_DEV_PILOT:
    if PILOT_OUT.exists() and any(PILOT_OUT.iterdir()):
        raise FileExistsError(f"Pilot output exists: {PILOT_OUT}. Choose a new folder name.")
    subprocess.run([
        sys.executable, str(PKG / "run_b1v3_development.py"),
        "--out", str(PILOT_OUT),
        "--dev-count", "4",
        "--score-cal-count", "12",
        "--tau-cal-count", "12",
        "--score-cal-rounds", "200",
        "--tau-cal-rounds", "200",
        "--rounds", "200",
    ], cwd=str(PKG), check=True)
    print("Pilot development completed. It is not the final test.")
else:
    print("Pilot skipped.")

## 4. Full development — اختيار معاملات PH وK من بيانات التطوير

في وضع `STAGE = "development"` تُشغّل خمس إعدادات PH فقط، بينما تُشغّل جميع خطوط Fixed-K مرة واحدة، لأن نتائجها لا تعتمد على معاملات PH. جميع التشغيلات تستخدم البذور نفسها وجداول الجلسات وهويات المهاجمين نفسها لضمان المقارنة الزوجية.

الإعدادات الافتراضية: 40 بذرة تطوير، و60 بذرة score calibration، و60 بذرة threshold calibration، و200 جولة. لا تدخل هذه البذور ضمن الاختبار النهائي.

In [ ]:
if RUN_FULL_DEVELOPMENT:
    if DEV_OUT.exists() and any(DEV_OUT.iterdir()):
        raise FileExistsError(f"Full development output exists: {DEV_OUT}. Choose a new folder name; do not overwrite silently.")
    subprocess.run([
        sys.executable, str(PKG / "run_b1v3_development.py"),
        "--out", str(DEV_OUT),
        "--dev-count", "40",
        "--score-cal-count", "60",
        "--tau-cal-count", "60",
        "--score-cal-rounds", "200",
        "--tau-cal-rounds", "200",
        "--rounds", "200",
    ], cwd=str(PKG), check=True)
    print("Full development completed:", DEV_OUT)
else:
    print("Full development skipped. Set RUN_FULL_DEVELOPMENT=True to execute it.")

## 5. اختيار معاملات PH وK-best وحساب حجم العينة

هذه الخلية تستخدم **Full development فقط**. تنتج إعدادات مجمدة وخطة بذور اختبار تستهدف اكتشاف `Delta J = 0.05` بقوة مخططة 90% بعد مراعاة عائلة الاختبارات الأساسية. هذه الخطة تقدير من بيانات التطوير وليست achieved power.

In [ ]:
SELECTION_FILE = DEV_OUT / "B1V3_selection.json"

if RUN_SELECTION:
    if not (DEV_OUT / "B1V3_development_protocol.json").exists():
        raise FileNotFoundError("Full development folder/protocol is missing. Run Full development first.")
    subprocess.run([
        sys.executable, str(PKG / "select_b1v3_development.py"),
        "--dev-root", str(DEV_OUT),
        "--meaningful-delta", "0.05",
        "--alpha-family", "0.05",
        "--target-power", "0.90",
        "--pfa-guardrail", "0.075",
        "--ffr-guardrail", "0.10",
    ], cwd=str(PKG), check=True)

if SELECTION_FILE.exists():
    selection = json.loads(SELECTION_FILE.read_text())
    print("Selected PH:", selection["selected_ph"])
    print("Selected fixed K:", selection["selected_fixed_k"])
    print("Sample-size plan:", selection["sample_size_plan"])
else:
    print("Selection file not yet available. Run Full development and enable RUN_SELECTION.")

## 6. فحص اختيار التطوير قبل الاختبار النهائي

راجع `B1V3_PH_development_grid.csv` و`B1V3_fixedK_development_grid.csv` و`B1V3_sample_size_plan.json`. إذا لم ينجح أي مرشح في حواجز PFA/FFR فسيذكر ملف الاختيار ذلك. لا تتجاوز التحذير لإجبار تشغيل اختبار ذي تصميم غير ملائم.

In [ ]:
if (DEV_OUT / "B1V3_PH_development_grid.csv").exists():
    import pandas as pd
    display(pd.read_csv(DEV_OUT / "B1V3_PH_development_grid.csv"))
    display(pd.read_csv(DEV_OUT / "B1V3_fixedK_development_grid.csv"))
    power_plan = json.loads((DEV_OUT / "B1V3_sample_size_plan.json").read_text())
    display(pd.DataFrame([power_plan]))
else:
    print("Development summaries not available yet.")

## 7. Final test — الاختبار المستقل المجمد

غيّر `STAGE` إلى `final` بعد إكمال development ومراجعة `B1V3_selection.json` و`B1V3_sample_size_plan.json`. يستخدم الاختبار إعداد PH وK المختارين في التطوير، ومجموعات معايرة جديدة، وبذور اختبار مستقلة. لا يضبط أي معامل أو عتبة على نتائج الاختبار النهائي.

In [ ]:
if RUN_FINAL_TEST:
    if not SELECTION_FILE.exists():
        raise FileNotFoundError("B1V3_selection.json is missing. Complete Full development and selection first.")
    if FINAL_OUT.exists() and any(FINAL_OUT.iterdir()):
        raise FileExistsError(f"Final output exists: {FINAL_OUT}. Choose a new final output folder; never overwrite an inspected test run.")
    subprocess.run([
        sys.executable, str(PKG / "run_b1v3_final.py"),
        "--selection", str(SELECTION_FILE),
        "--out", str(FINAL_OUT),
        "--score-cal-count", str(FINAL_SCORE_CAL_COUNT),
        "--tau-cal-count", str(FINAL_TAU_CAL_COUNT),
    ], cwd=str(PKG), check=True)
    print("Final simulation completed:", FINAL_OUT)
else:
    print("Final test skipped. Enable RUN_FINAL_TEST only after reviewing the frozen selection.")

## 8. تحليل الاختبار النهائي

التحليل يستخدم البذرة كعينة مستقلة. المقارنات الأساسية هي XS-PH مقابل XS-fixed20 وXS-fixedK-best في الجلسات المتغيرة عند SNR −9 و−6 dB. تقارن النتائج عند عتبات عُيّرت على quantile اسمي 0.95 في بيانات H0 مستقلة؛ لذلك تسمى **calibration-target PFA**، ولا يُفترض أن achieved PFA في الاختبار يساوي 0.05 تماماً.

In [ ]:
if RUN_FINAL_ANALYSIS:
    if not (FINAL_OUT / "v2_seed_level.csv.gz").exists():
        raise FileNotFoundError("Final test results are missing. Run Final test first.")
    subprocess.run([
        sys.executable, str(PKG / "analyze_b1v3.py"),
        "--results", str(FINAL_OUT),
        "--selection", str(SELECTION_FILE),
        "--target-pfa", "0.05",
        "--n-permutations", "100000",
        "--n-bootstrap", "20000",
    ], cwd=str(PKG), check=True)
    print("Final analysis completed:", FINAL_OUT)
else:
    print("Final analysis skipped. Enable RUN_FINAL_ANALYSIS after Final test.")

## 9. عرض النتائج الأساسية

لا تعتبر وجود ملف النتائج وحده دليلاً على تفوق PH. اقرأ جدول المقارنات الأساسية، وشرط clean PFA/FFR، ودقة الحدود بصورة منفصلة.

In [ ]:
import pandas as pd

if (FINAL_OUT / "B1V3_primary_paired_tests.csv").exists():
    print("=== Primary paired tests ===")
    display(pd.read_csv(FINAL_OUT / "B1V3_primary_paired_tests.csv"))

    print("=== Performance summary: variable sessions ===")
    perf = pd.read_csv(FINAL_OUT / "B1V3_performance_summary.csv")
    display(perf[perf.session_spec == "random_balanced:13,20,30,50"])

    print("=== Boundary summary at 2-round maximum delay ===")
    bound_path = FINAL_OUT / "B1V3_boundary_summary.csv"
    if bound_path.exists():
        bound = pd.read_csv(bound_path)
        display(bound[(bound.session_spec == "random_balanced:13,20,30,50") &
                      (bound.max_allowed_delay == 2)])

    print("=== No-transition control ===")
    display(pd.read_csv(FINAL_OUT / "B1V3_no_change_control.csv"))

    conclusion = json.loads((FINAL_OUT / "B1V3_final_conclusion_aid.json").read_text())
    print("Performance-improvement claim supported:",
          conclusion["claim_ph_performance_improvement_supported"])
    print((FINAL_OUT / "B1V3_interpretation_report.md").read_text())
else:
    print("Final analysis outputs are not available yet.")

## 10. ضغط النتائج لحفظها خارج جلسة Kaggle

In [ ]:
if FINAL_OUT.exists() and any(FINAL_OUT.iterdir()):
    archive_path = shutil.make_archive(
        str(WORK_ROOT / "B1V3_final_results"),
        "zip",
        root_dir=FINAL_OUT,
    )
    print("Final results archive:", archive_path)
else:
    print("No final results folder to archive yet.")

## مخرجات يجب الاحتفاظ بها

احتفظ مع النتائج النهائية بملفات التطوير `B1V3_selection.json` و`B1V3_sample_size_plan.json`، وملفات `v2_manifest.json` و`v2_seed_ledger.json` وبصمات المصدر. لا تنقل استنتاجات Pilot إلى Abstract باعتبارها نتائج تأكيدية. النتائج الإيجابية في الاختبار الجديد يجب أن تكون متوافقة مع المقارنات الأساسية وفواصل الثقة وتصحيح Holm وحواجز الاستبعاد النظيف.